# EXP-01-R · Quantus FC on real Adult records

**질문:** 실제 Adult 데이터와 학습된 모델에서도, 설명은 그대로 두고 묶음 크기 k를 바꾸면 population FC가 바뀔까? 아니면 B개 묶음으로 계산한 유한 표본 점수만 달라질까?

We use OpenXAI's processed **Adult Income** records, fit one logistic regression on training data, explain its fixed **high-income log-odds** with **Captum IG**, and obtain metric scores from **official Quantus 0.6.0 FC/FE**. Labels are income outcomes; the attribution reference is derived from this model, not a human or causal explanation label.

Adult + logistic regression + IG appears in [OpenXAI, NeurIPS 2022, Table 3](https://papers.nips.cc/paper/2022/file/65398a0eba88c9b4a1c38ae405b125ef-Paper-Datasets_and_Benchmarks.pdf). Our contract and scores differ from that paper's experiment. [Compact design](https://github.com/theguardianofthebed/xai-metric-behavior/blob/exp01-atlas-protocol/docs/EXP-01-R.md).


## 1. Prepare an isolated CPU environment

**Colab:** open the latest notebook in a fresh CPU runtime. This cell clones `exp01-atlas-protocol` when needed and creates or repairs `.venv`, including the missing-`ensurepip` case. An existing checkout is not silently updated; if it predates EXP-01-R, use a fresh runtime or update it explicitly. **Local Jupyter:** work in the repository or set `REPO_DIR`.

The notebook kernel stays separate. Quantus, Captum, scikit-learn and CPU PyTorch are installed only in the experiment environment. Initial setup can take several minutes. The same CLI engine is used below.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, tomllib

REPOSITORY = "https://github.com/theguardianofthebed/xai-metric-behavior.git"
REF = "exp01-atlas-protocol"
REPO_DIR = None  # Optionally set the path of an existing checkout.

if REPO_DIR:
    repo = Path(REPO_DIR).expanduser().resolve()
else:
    repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "configs/exp01r_smoke.yaml").exists()
                 and (p / "pyproject.toml").exists()), None)
    if repo is None:
        repo = (Path("/content") if Path("/content").exists() else Path.cwd()) / "xai-metric-behavior"
        if not repo.exists():
            subprocess.run(["git", "clone", "--branch", REF, REPOSITORY, str(repo)], check=True)
if not (repo / "configs/exp01r_smoke.yaml").exists():
    raise RuntimeError("This checkout predates EXP-01-R. Use a fresh Colab runtime or explicitly update the exp01-atlas-protocol branch, then rerun this cell.")
project = tomllib.loads((repo / "pyproject.toml").read_text())["project"]
if not any(req.lower().startswith("quantus") for req in project["dependencies"]):
    raise RuntimeError("This checkout predates the Quantus migration. Update it explicitly or use a fresh clone.")
os.chdir(repo)
def run_setup(command, stage):
    print(stage, flush=True)
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f"{stage} failed (exit {result.returncode}).\n"
                           f"Command: {command!r}\n{result.stdout}")
    return result


def isolated_environment_ready(env_dir, env_python):
    if not env_python.exists():
        return False
    probe = subprocess.run(
        [str(env_python), "-I", "-c", "import sys,json; print(json.dumps([sys.prefix,sys.base_prefix]))"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if probe.returncode:
        return False
    try:
        prefix, base = json.loads(probe.stdout)
        config = (env_dir / "pyvenv.cfg").read_text().lower()
        return (Path(prefix).resolve() == env_dir.resolve() and prefix != base
                and "include-system-site-packages = false" in config)
    except (ValueError, OSError):
        return False


env_dir = repo / ".venv"
env_python = env_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if Path(sys.prefix).resolve() == env_dir.resolve():
    raise RuntimeError("Select a notebook kernel outside the experiment's .venv before setup; "
                       "experiment installs must not modify the running kernel.")
if not isolated_environment_ready(env_dir, env_python):
    # Some notebook runtimes lack ensurepip. Create the venv without invoking it.
    run_setup([sys.executable, "-m", "venv", "--without-pip", str(env_dir)], "Prepare isolated environment")
if not isolated_environment_ready(env_dir, env_python):
    raise RuntimeError(f"The isolated interpreter is not usable: {env_python}")
pip_probe = subprocess.run([str(env_python), "-m", "pip", "--version"],
                           text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if pip_probe.returncode:
    # Kernel pip manages the target interpreter; nothing is installed in the kernel.
    # This also repairs a partial .venv left by the old setup cell.
    kernel_pip = run_setup([sys.executable, "-m", "pip", "--help"], "Check kernel pip")
    if "--python" not in kernel_pip.stdout:
        raise RuntimeError("Setup requires kernel pip >=22.3 with --python support. "
                           "Use a fresh Colab runtime or a notebook kernel with a recent pip.")
    run_setup([sys.executable, "-m", "pip", "--python", str(env_python), "install", "--quiet", "pip"],
              "Bootstrap pip in the isolated environment")
run_setup([str(env_python), "-m", "pip", "--version"], "Verify experiment pip")
run_setup([str(env_python), "-m", "pip", "install", "--quiet", "torch==2.10.0",
           "--index-url", "https://download.pytorch.org/whl/cpu"], "Install CPU PyTorch")
run_setup([str(env_python), "-m", "pip", "install", "--quiet", "-e", f"{repo}[exp01r]"],
          "Install Quantus, Captum and EXP-01-R dependencies")
version_query = (
    "import sys,json,importlib.metadata as m; "
    "print(json.dumps({'Python':sys.version.split()[0], "
    "**{p:m.version(p) for p in ['xai-metric-behavior','quantus','captum','scikit-learn','torch','numpy','scipy']}}))"
)
versions = json.loads(subprocess.check_output([str(env_python), "-c", version_query], text=True))
print("Repository:", repo)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Notebook Python:", sys.version.split()[0])
print("Experiment interpreter:", env_python)
print(json.dumps(versions, indent=2))


## 2. Read the contract and select smoke

- **Data:** 36,177 training / 9,045 test records. Six numeric inputs (age, census sampling weight, education level, capital gain/loss, weekly hours) and seven binary indicators (sex, workclass, marital status, occupation, relationship, race, country). Label 1 means income >$50K.
- **Version:** immutable author CSVs linked by official OpenXAI code; file hashes and columns are checked. The planned Dataverse V3 download returned 403, so identity with V3 is unverified. This source change is recorded before measurement.
- **Fixed:** train-only StandardScaler, one logistic model including intercept, high-income log-odds target, zero baseline in standardized coordinates, signed Captum IG. Zero means training feature means in raw encoded coordinates, which may be fractional categories.
- **Candidates:** IG, A=one-coordinate cyclic shift, B=two-coordinate shift. The shifts preserve attribution values and norms. A and B have no guaranteed ordering; each input has its own population margin.
- **Varied:** k features per sampled group, B groups per score. N counts input records; R counts evaluator repetitions. No scores are replaced by NumPy.
- **Reserved:** 512 disjoint test row IDs and k={2,7,11}. Smoke and development do not evaluate them. Confirmation is not implemented in this notebook.

Smoke: N=4, R=4, k={1,5,10,12}, B={5,25,100}. Inspect it before any development run. A development run reuses the saved fitted model in the same data cache. Its larger grid can take much longer.


In [ ]:
PROFILE = "smoke"  # Use "development" only after reviewing smoke.
if PROFILE not in {"smoke", "development"}:
    raise ValueError("Reserved confirmation is blocked until a separate freeze workflow is implemented.")
config_path = repo / "configs" / f"exp01r_{PROFILE}.yaml"
data_dir = repo / "data" / "cache" / "openxai-adult"
print(config_path.read_text())


## 3. Run the experiment

The CLI verifies/downloads the pinned CSVs, reserves row IDs, fits or reuses the frozen training pipeline, generates actual Captum IG, and checks each coordinate and completeness. It then runs official Quantus calls, checks recorded masks/U/V, and writes raw returns, failures and results.

An IG or instrumentation gate failure stops the run with a saved failure record. Upstream validation exceptions, undefined scores and near-constant warnings remain visible; they are never turned into zero or replaced by new samples. Every run gets a new output directory.


In [ ]:
from datetime import datetime, timezone
from time import perf_counter

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = repo / "results" / f"exp01r-{PROFILE}-{stamp}"
started = perf_counter()
subprocess.run([str(env_python), "-m", "xai_metric_behavior", "run-real",
                "--config", str(config_path), "--output", str(output_dir),
                "--data-dir", str(data_dir)], cwd=repo, check=True)
print("Saved:", output_dir)
print(f"Elapsed: {perf_counter() - started:.1f} seconds")


## 4. Read the answer to the research question

**Population/reference versus k:** the reference concerns every allowed size-k group for one input. Enumeration uses actual replacements, so its comparison is independent of merely repeating a horizontal formula. **Finite Quantus scores:** points/means summarize only R repeated finite samples at a specified B; they need not coincide with the reference.

**Error versus B:** zero error means agreement with the population reference. RMSE measures finite estimation error, not explanation quality. Repeats belong to the same input, and summary inputs are weighted equally. A smaller error in one smoke grid is not proof that every score improves as B increases. Always read eligible/all-attempt counts and failures with the curves.

The printed report below is generated from this run's saved scores and checks. Smoke is a small implementation check; it is not held-out confirmation.


In [ ]:
import json
from IPython.display import display, Markdown, Image

print((output_dir / "rq_answer.txt").read_text())
manifest = json.loads((output_dir / "manifest.json").read_text())
print(json.dumps({key: manifest[key] for key in
                 ["status", "phase", "run_id", "counts", "validation"]
                 if key in manifest}, indent=2))
figure_names = ["rq_population_vs_estimates.png", "rq_error_vs_budget.png",
                "rq_population_enumeration.png", "rq_failure_fractions.png"]
for name in figure_names:
    figure = output_dir / "figures" / name
    display(Markdown(f"### {figure.stem.replace('_', ' ')}"))
    display(Image(filename=str(figure)))


## 5. Inspect all attempts and keep the evidence

`failures.csv` preserves reasons; an empty table means no recorded failures in this run, not a universal guarantee. `scores.csv` retains actual Quantus values, while `analysis_eligible` identifies the conditional analysis set. The data/model/partition records and IG checks show which real inputs and fixed model were used.

Keep the **complete output directory and fitted model cache** when continuing to development. The 512 reserved rows and reserved k values remain unused. One model, encoded-coordinate replacement and a tiny smoke sample do not establish causal explanations, training-seed stability or general metric reliability.


In [ ]:
import csv

for name in ["data_manifest.json", "model.json", "partition.json"]:
    path = output_dir / name
    print(name, "saved" if path.exists() else "see manifest artifact map")
for name in ["ig_checks.csv", "failures.csv", "scores.csv"]:
    path = output_dir / name
    if not path.exists():
        continue
    with path.open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    print(f"\n{name}: {len(rows)} rows; first 3:")
    for row in rows[:3]:
        print(row)
print("\nKeep:", output_dir)
print("Fitted model cache:", data_dir)


## Research Learning Summary

**필요성 →** 합성 예제에서 확인한 sampling 기제가 실제 입력·학습 모델·XAI 구현에서도 관찰되는지 점검한다.  
**직관적인 예시/이해 →** 한 사람의 IG 설명을 고정하고 Quantus가 뽑는 feature 묶음 크기와 횟수만 바꾼다.  
**공식적인 정의 →** additive log-odds에서 population FC는 좌표별 attribution과 모델 기여도의 correlation이고, 실제 Quantus score는 B개 묶음으로 얻은 값이다.  
**근거/그게 되는 이유 →** additivity와 uniform subset sampling의 공통 계수가 population Pearson에서 상쇄된다. 실제 데이터는 전처리·학습·수치·표현 제약을 더하며, 작은 smoke는 이 범위의 구현 확인이지 보편적인 metric 신뢰성 증명이 아니다.
